# FBN1 Cohort Analysis: Correlation

In this tutorial, we analyze a cohort of phenopackets associated with the gene **FBN1**. The example data is retrieved from the [phenopacket store](https://github.com/monarch-initiative/phenopacket-store).

Using this dataset, we demonstrate a complete workflow with **phenocor**, including:

- dataset construction
- correlation analysis

We recommend running this tutorial in a **Jupyter notebook** for interactive exploration and visualization, although it can also be executed as a standard Python script.


## 1. Load phenopackets

We start by loading phenopackets for the **FBN1** cohort:

### Requirements

Install phenocor and the toolkit used to retrieve the example data:

```bash
pip install phenocor phenopacket-store-toolkit
```

The first data download requires an internet connection.

In [13]:
from ppktstore.registry import configure_phenopacket_registry
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store() as ps:
        phenopackets = [
            p for p in ps.iter_cohort_phenopackets("FBN1")
        ]

print(f"Loaded {len(phenopackets)} phenopackets")

Loaded 143 phenopackets


The registry provides access to Phenopacket Store cohorts through iter_cohort_phenopackets().

> **Note：**  
> You can also use your own phenopacket data. As long as your data follows the [GA4GH Phenopacket schema](https://phenopacket-schema.readthedocs.io/en/latest/index.html), it can be directly used in the workflow.

## 2. Build the dataset

Next, we convert phenopackets into a structured dataset suitable for downstream statistical analysis. 

In [14]:
from phenocor import PhenotypeDatasetBuilder

dataset = PhenotypeDatasetBuilder(phenopackets).build()

The resulting `dataset` is now ready for correlation.


## 3. Correlation analysis

Next, we compute pairwise correlations between HPO features across individuals in the cohort.

In [15]:
from phenocor import HPOCorrelationAnalyzer

analyzer = HPOCorrelationAnalyzer(dataset=dataset)

results = analyzer.compute_correlation_matrix()

results.correlation_results.head()

Calculating pairwise correlations: 100%|██████████| 728/728 [00:01<00:00, 486.47it/s]


,HPO_A,HPO_B,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,HPO_A_label,HPO_B_label,n_pmids,pmids
0,HP:0004322,HP:0004942,-0.827131,2.049015e-14,9.773799e-12,7,34,33,0,74,Short stature,Aortic aneurysm,5,10756346;11175294;12203992;20375004;21683322
1,HP:0000098,HP:0004942,0.796637,1.940118e-13,4.627183e-11,45,7,1,26,79,Tall stature,Aortic aneurysm,13,10756346;11175294;12203992;20375004;20979188;2...
2,HP:0000098,HP:0004322,-0.685681,5.424742e-12,8.625340e-10,16,33,37,0,86,Tall stature,Short stature,5,10756346;11175294;12203992;20375004;21683322
3,HP:0000098,HP:0002616,0.646903,1.766457e-11,2.106500e-09,70,7,10,26,113,Tall stature,Aortic root aneurysm,18,10756346;11175294;12203992;20375004;20979188;2...
4,HP:0002616,HP:0004322,-0.634615,2.474657e-10,2.360823e-08,19,33,33,0,85,Aortic root aneurysm,Short stature,5,10756346;11175294;12203992;20375004;21683322


This step identifies positive and negative associations between HPO terms.

In the result table:

- `HPO_A` and `HPO_B` are the two HPO terms being compared  
- `correlation` indicates the strength and direction of association  
- `adj_p_value` is the Benjamini–Hochberg-adjusted p-value.  

Interpretation:

- **positive correlation** → the two phenotypes tend to appear together  
- **negative correlation** → the phenotypes tend to occur in different individuals  
- **values near zero** → little or no association  

Detailed descriptions of parameters and output columns are provided in the **Usage** section.

## 4. Visualize correlation results

To better interpret these relationships, we visualize them as a heatmap.

In [16]:
import os
from IPython.display import HTML

STATIC_DIR = "_static" 
os.makedirs(STATIC_DIR, exist_ok=True)

def plotly_html_link(fig, filename, link_text="View interactive heatmap"):
    """
    Generate Plotly HTML file and return an RTD-friendly link.
    """
    filepath = os.path.join(STATIC_DIR, filename)
    fig.write_html(filepath, include_plotlyjs="cdn", full_html=True)
    return HTML(f'<p>To view the interactive plot, click the link below:</p>'
                f'<a href="_static/{filename}?v=0.2.1" target="_blank">{link_text}</a>')

In [17]:
fig1 = results.plot_correlation_heatmap_with_significance(
    title="Cohort FBN1",
)

In [18]:
plotly_html_link(fig1, "correlation_heatmap.html", link_text="Click here to view interactive correlation heatmap")

This visualization highlights the strongest and most statistically significant associations between phenotypic features.

- Strong positive correlations indicate phenotypes that frequently co-occur
- Strong negative correlations indicate phenotypes that tend to occur in different individuals

> **Note：**  
> Hover over the heatmap to see detailed information for each phenotype pair.  
> Increasing the adjusted p-value threshold includes more pairs. Increasing the minimum absolute Phi coefficient retains stronger associations..  

## Summary

In this tutorial, we:

- Loaded phenopacket data  
- Constructed a structured dataset  
- Explored phenotype–phenotype relationships using correlation  

Together, these steps demonstrate phenotype correlation analysis within the FBN1 cohort.

For additional usage patterns and parameter options, see the **Usage** section.